# 🏎️ APEX AI — Production Fine-Tuning & Live Tunnel Server (v2.2)

This notebook delivers an end-to-end, zero-error pipeline on **Google Colab's Free T4 GPU**:
1. **Fine-tunes Llama-3-8B** using **Unsloth (QLoRA 4-bit)** on the updated F1 Senior Technical Director dataset.
2. **Eliminates Truncation & Word Limits**: `max_new_tokens=850` with repetition penalties and anti-looping safeguards.
3. **Forces Definitive Technical Conclusions**: Telemetry numbers, tyre degradation rates, aero operating windows, and closing summaries.
4. **Launches a Live Ngrok Tunnel**: Directly bridges your local `APEX CLI` terminal to Colab.

---

## Step 1: Install Dependencies (Fast 4-bit Unsloth Stack)

In [ ]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes triton
!pip install -q fastapi uvicorn pyngrok nest-asyncio

## Step 2: Load 4-bit Llama-3-8B & Attach LoRA Adapters

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto-detects float16 for Tesla T4 GPU
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Attach LoRA parameter-efficient adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

print("✓ Model and LoRA adapters initialized successfully!")

## Step 3: Pull Latest Clean Training Dataset from GitHub

In [ ]:
# Clone or pull fresh repo
import os
if not os.path.exists("repo"):
    !git clone https://github.com/bhxvxsh/yeah-def-my-nyc-project.git repo
else:
    !cd repo && git pull origin master

from datasets import load_dataset

data_path = "repo/APEX_AI/03_Training_Dataset/f1_technical_alpaca.json"
dataset = load_dataset("json", data_files=data_path, split="train")

prompt_template = """Below is an instruction from a Formula 1 team principal or research analyst. Provide an uncompromisingly technical, authoritative, and complete engineering evaluation. Do NOT output generic python feature templates. Provide real telemetry benchmarks, degradation numbers, and always conclude with a definitive technical summary.

### Instruction:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = prompt_template.format(instruction, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

formatted_dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"✓ Loaded {len(dataset)} verified technical F1 samples.")

## Step 4: Fine-Tune Model (Quick ~10-15 Min Run)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()
print("✓ Fine-tuning complete! Activating fast inference mode...")
FastLanguageModel.for_inference(model)
model.eval()

## Step 5: Start Live Fast-API Server & Ngrok Tunnel for APEX CLI

> **Important:** Set your free Ngrok token below if you have one (`ngrok.set_auth_token("YOUR_TOKEN")`). If not, the public tunnel will start directly.

In [ ]:
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
from pyngrok import ngrok
import traceback

# Ensure model is in inference mode
FastLanguageModel.for_inference(model)
model.eval()

app = FastAPI(title="APEX AI Inference Server")

class QueryRequest(BaseModel):
    query: str

@app.get("/health")
def health():
    return {"status": "ok", "model": "APEX-AI-Senior-Race-Strategist"}

@app.post("/ask")
def ask(req: QueryRequest):
    try:
        user_text = req.query.strip()
        print(f"\n[User Query Received]: {user_text}")
        
        prompt = (
            "Below is an instruction from a Formula 1 team principal or research analyst. "
            "Provide an uncompromisingly technical, authoritative, and complete engineering evaluation. "
            "Do NOT output generic python feature templates. Provide real telemetry benchmarks, degradation numbers, "
            "and always conclude with a definitive technical summary.\n\n"
            f"### Instruction:\n{user_text}\n\n"
            "### Response:\n"
        )
        
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        
        with torch.inference_mode():
            outputs = model.generate(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=850,        # Increased token ceiling: eliminates truncation
                temperature=0.6,           # Controlled, factual outputs
                top_p=0.9,
                repetition_penalty=1.15,   # Prevents looping on lists of features
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
        answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        
        print(f"[Generated Response Length]: {len(answer)} chars")
        return {"response": answer}
        
    except Exception as e:
        err_msg = f"{type(e).__name__}: {str(e)}\n{traceback.format_exc()}"
        print(f"[ERROR]: {err_msg}")
        return {"response": f"Colab Server Error: {str(e)}"}

# Optional: Set your token if you have one
# ngrok.set_auth_token("YOUR_FREE_NGROK_TOKEN")

# Kill any previous sessions
ngrok.kill()
public_url = ngrok.connect(8000).public_url

print("\n" + "=" * 70)
print(f"🚀 YOUR LIVE APEX AI COLAB URL: {public_url}")
print(f"👉 In your local APEX CLI, type: /connect {public_url}")
print("=" * 70 + "\n")

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)